In [1]:
records = [
    {
        "patient_id": "P1",
        "admission": "2024-01-10",
        "discharge": "2024-01-15",
        "diagnosis": "E11",
        "age": 65,
        "cost": 1200
    },
    {
        "patient_id": "P2",
        "admission": "2024-02-01",
        "discharge": "2024-02-03",
        "diagnosis": "I10",
        "age": 50,
        "cost": 800
    },
    {
        "patient_id": "P1",
        "admission": "2024-02-20",
        "discharge": "2024-02-28",
        "diagnosis": "E11",
        "age": 65,
        "cost": 2000
    },
    {
        "patient_id": "P3",
        "admission": "2024-03-01",
        "discharge": None,
        "diagnosis": "J18",
        "age": 72,
        "cost": 1500
    },
    {
        "patient_id": "P4",
        "admission": "2024-03-10",
        "discharge": "2024-03-05",
        "diagnosis": "E11",
        "age": 40,
        "cost": 500
    }
]

In [19]:
from datetime import datetime
from collections import Counter

today = datetime(2024, 6, 1)

def build_report(records: list[dict]) -> dict:
    total_delay = 0
    nb_completed_stays = 0
    diagnosis = []
    strange_record = []
    patients = {}

    for record in records:
        patient_id = record["patient_id"]
        admi = datetime.strptime(record["admission"], "%Y-%m-%d")
        disc = None

        if record["discharge"]:
            disc = datetime.strptime(record["discharge"], "%Y-%m-%d")
        else:
            duration_record = (today - admi).days
            if duration_record > 90:
                strange_record.append(record)
                continue

        if disc and disc < admi:
            strange_record.append(record)
            continue

        # ---- calcul durée séjour terminé uniquement ----
        if disc:
            duration_record = (disc - admi).days
            total_delay += duration_record
            nb_completed_stays += 1
        else:
            duration_record = None

        # ---- patient infos ----
        if patient_id not in patients:
            patients[patient_id] = {
                "cost": 0,
                "nb_hospi": 0,
                "admissions": []
            }

        patients[patient_id]["cost"] += record["cost"]
        patients[patient_id]["nb_hospi"] += 1

        if disc:
            patients[patient_id]["admissions"].append((admi, disc))

        diagnosis.append(record["diagnosis"])

    # ---- diagnostic le plus fréquent ----
    diag_counter = Counter(diagnosis)
    most_common = diag_counter.most_common(1)
    most_common_diag = most_common[0][0] if most_common else None

    # ---- taux de readmission ≤ 30 jours ----
    readmitted_patients = 0

    for patient_id in patients:
        admissions = patients[patient_id]["admissions"]
        admissions.sort(key=lambda x: x[0])  # tri par date admission

        readmitted = False

        for i in range(1, len(admissions)):
            prev_discharge = admissions[i-1][1]
            current_admission = admissions[i][0]

            if (current_admission - prev_discharge).days <= 30:
                readmitted = True
                break

        if readmitted:
            readmitted_patients += 1

    readmission_rate = (
        readmitted_patients / len(patients)
        if patients else 0
    )

    return {
        "visit_duration": (
            total_delay / nb_completed_stays
            if nb_completed_stays else 0
        ),
        "patient_data": patients,
        "nb_strange_record": len(strange_record),
        "most_common_diag": most_common_diag,
        "readmission_rate_30": readmission_rate
    }
        
build_report(records)

{'visit_duration': 5.0,
 'patient_data': {'P1': {'cost': 3200,
   'nb_hospi': 2,
   'admissions': [(datetime.datetime(2024, 1, 10, 0, 0),
     datetime.datetime(2024, 1, 15, 0, 0)),
    (datetime.datetime(2024, 2, 20, 0, 0),
     datetime.datetime(2024, 2, 28, 0, 0))]},
  'P2': {'cost': 800,
   'nb_hospi': 1,
   'admissions': [(datetime.datetime(2024, 2, 1, 0, 0),
     datetime.datetime(2024, 2, 3, 0, 0))]}},
 'nb_strange_record': 2,
 'most_common_diag': 'E11',
 'readmission_rate_30': 0.0}